# 01 · Una run

Cosa è successo in una sola run: carico generato da wrk2, segnali per servizio, controlli di validità.

I segnali sono quelli del builder (`anomaly/build.py`): griglia di `STEP` secondi, rate dei counter calcolati sui timestamp reali, celle vuote dove cAdvisor non ha campioni. Il tempo è in minuti dall'iniezione; nelle sessioni pulite l'iniezione è un noop e segna solo la fine del settle.

In [ ]:
import sys
import warnings
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "anomaly" / "build.py").exists())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from anomaly import explore as ex

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_colwidth", 90)

In [ ]:
DATA = Path("/home/tecnico/datasets/pilot")  # output di collect_dataset (--out), una sua sottocartella, o datasets/trials
RUN = None       # id di una run della lista stampata sotto; None prende la prima
STEP = 30        # passo della griglia in secondi
SERVICES = None  # servizi da disegnare; None prende il target e l'entry point

runs = ex.find_runs(DATA)
print(f"{len(runs)} run valide in {DATA}")
for run_id in runs:
    print(" ", run_id)

In [ ]:
run_id = RUN or next(iter(runs))
run = ex.load(runs[run_id], STEP)
L = run.labels
minutes = run.t_rel / 60
print(f"{run_id}\n  tipo {L['kind']}, split {L.get('split')}, guasto: {L['fault_type']}, target: {L['target']}")
print(f"  {run.x.shape[0]} passi da {STEP} s, {len(run.services)} servizi, {len(run.features)} feature")
print(f"  cambio di carico più vicino all'iniezione: {ex.gap_to_injection(run):.0f} s")
for name, value in L["checks"].items():
    if name != "window":
        print(f"  {name}: {'ok' if value[0] else 'FALLITO'}  {value[1]}")

## Carico

Rate e mix di wrk2 a ogni passo. Linea rossa tratteggiata: iniezione; linea sottile: fine dell'iniezione. Sotto, un segmento di wrk2 per riga: rate erogato, p99 e risposte non 2xx.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(13, 4), sharex=True)
ax[0].step(minutes, np.where(run.rate >= 0, run.rate, np.nan), where="post")
ax[1].step(minutes, np.where(run.mix >= 0, run.mix, np.nan), where="post", color="tab:orange")
ax[0].set_ylabel("req/s")
ax[1].set_ylabel("mix")
ax[1].set_xlabel("minuti dall'iniezione")
for a in ax:
    a.axvline(0, color="red", ls="--")
    a.axvline((L["t_injected"] - L["t_inject"]) / 60, color="red", lw=0.6)
plt.show()

segments = pd.DataFrame(L["segments"])
if len(segments):
    segments.insert(0, "inizio_min", ((segments.start - L["t_inject"]) / 60).round(1))
    display(segments.drop(columns="start"))

## Tutti i segnali

Un pannello per feature: righe = servizi, colonne = tempo. Colore = z robusto rispetto al settle, `(x − mediana) / IQR`, tagliato a ±8. Bianco = nessun campione. Nelle sessioni pulite i salti di colore che seguono i cambi di carico sono normali: è proprio ciò che il modello deve imparare a ignorare.

In [ ]:
ref = ex.settle_mask(run)
if not ref.any():  # la prova di capacità ha 30 s di settle: nessun passo di griglia ci cade dentro
    ref = ex.steady_mask(run)
    print("Settle troppo corto per fare da riferimento: uso tutti i passi a carico stabile")
z = ex.robust_z(run.x, ref)
cols = 3
rows = -(-len(run.features) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(18, (0.2 * len(run.services) + 1) * rows), sharex=True, constrained_layout=True)
extent = [minutes[0], minutes[-1], len(run.services) - 0.5, -0.5]
for j, (a, feature) in enumerate(zip(axes.flat, run.features)):
    image = a.imshow(np.clip(z[:, :, j].T, -8, 8), aspect="auto", cmap="RdBu_r", vmin=-8, vmax=8, extent=extent, interpolation="nearest")
    a.set_title(feature)
    a.set_yticks(range(len(run.services)), run.services, fontsize=7)
    a.axvline(0, color="k", lw=1)
for a in axes.flat[len(run.features):]:
    a.axis("off")
fig.colorbar(image, ax=axes, shrink=0.5, label="z robusto rispetto al settle")
plt.show()

## Serie di alcuni servizi

Valori assoluti: CPU in core, memoria in byte, rete in byte/s, restart al secondo.

In [ ]:
show = SERVICES or list(dict.fromkeys(ex.target_services(run) + [run.services[0]]))
features = ["cpu", "cpu_throttled", "mem_working_set", "threads", "rx_bytes", "tx_bytes", "restarts"]
fig, axes = plt.subplots(len(features), 1, figsize=(13, 2 * len(features)), sharex=True)
for a, feature in zip(axes, features):
    for service in show:
        a.plot(minutes, ex.series(run, service, feature), label=service)
    a.set_ylabel(feature)
    a.axvline(0, color="red", ls="--")
axes[0].legend(fontsize=8, loc="upper left")
axes[-1].set_xlabel("minuti dall'iniezione")
plt.show()

## Dati mancanti

Quota di passi senza campione dopo l'iniezione, per servizio e feature (0 = completo; 1 = la feature non esiste per quel servizio, per esempio il throttling su Social).

In [ ]:
after = run.t_rel > 0
pd.DataFrame(1 - run.mask[after].mean(axis=0), index=run.services, columns=run.features).round(2)